# Tutoriel 11

> 🧭 **Séance hors-programme, pour les curieux.** Cette séance ne fait pas partie de la matière évaluée. Elle reprend tout ce qui a été vu aux séances 7 à 10 et constitue un bon projet personnel pour qui souhaite aller plus loin.

# Manipuler les lignes et les colonnes d'une matrice 2D

L'exercice suivant demande de faire des opérations sur les lignes et les colonnes d'une matrice afin d'en modifier légèrement la taille, ce qui est nécessaire à l'implémentation de la méthode numérique. Nous allons donc nous exercer à ces manipulations avant de passer à l'application.

**Trois opérations suffisent**, et vous les connaissez déjà en 1D :

| Opération | Effet sur la taille | Quand l'utiliser |
|---|---|---|
| **dérivation** | on perd une cellule | quand on a besoin d'une dérivée |
| **moyenne** | on perd une cellule | quand on a besoin d'une valeur *entre* deux points |
| **troncation** | on perd deux cellules | quand on doit renoncer aux deux bords |

En 2D, il faut en plus choisir **selon quel axe** : le premier indice désigne la **ligne** (axe $y$), le second la **colonne** (axe $x$). Donc `M[:,1:]` travaille selon $x$, et `M[1:,:]` selon $y$.

In [1]:
import numpy as np

M = np.random.randint(0, 11, (3, 4))
print("M =\n", M, "\ntaille :", M.shape, " -> 3 lignes, 4 colonnes\n")

# 1) derivation : on perd une cellule selon l'axe de la difference
print("M[:,1:] - M[:,:-1]      ->", (M[:,1:]-M[:,:-1]).shape, " une COLONNE en moins")
print("M[1:,:] - M[:-1,:]      ->", (M[1:,:]-M[:-1,:]).shape, " une LIGNE en moins")

# 2) moyenne : la valeur ENTRE deux points, on perd aussi une cellule
print("0.5*(M[:,1:]+M[:,:-1])  ->", (0.5*(M[:,1:]+M[:,:-1])).shape, " comme la derivee")
print("0.5*(M[1:,:]+M[:-1,:])  ->", (0.5*(M[1:,:]+M[:-1,:])).shape, " comme la derivee")

# 3) troncation : on renonce a la premiere et a la derniere ligne / colonne
print("M[:,1:-1]               ->", M[:,1:-1].shape, " deux colonnes en moins")
print("M[1:-1,:]               ->", M[1:-1,:].shape, " deux lignes en moins")

M =
 [[9 2 3 0]
 [8 2 9 5]
 [7 9 0 4]] 
taille : (3, 4)  -> 3 lignes, 4 colonnes

M[:,1:] - M[:,:-1]      -> (3, 3)  une COLONNE en moins
M[1:,:] - M[:-1,:]      -> (2, 4)  une LIGNE en moins
0.5*(M[:,1:]+M[:,:-1])  -> (3, 3)  comme la derivee
0.5*(M[1:,:]+M[:-1,:])  -> (2, 4)  comme la derivee
M[:,1:-1]               -> (3, 2)  deux colonnes en moins
M[1:-1,:]               -> (1, 4)  deux lignes en moins


**Dérivation et moyenne donnent la même taille** : c'est ce qui permet de les multiplier entre elles, comme dans `D * (s[:,1:]-s[:,:-1])/dx` où `D` provient d'une moyenne.

Enchaînons-les comme dans un vrai modèle 2D, en surveillant la taille à chaque ligne :

In [2]:
ny, nx, dx, dy = 40, 60, 1.0, 1.0
S = np.random.random((ny, nx))

qx = - 0.5*(S[:,1:]+S[:,:-1]) * ( S[:,1:] - S[:,:-1] )/dx    # flux selon x
qy = - 0.5*(S[1:,:]+S[:-1,:]) * ( S[1:,:] - S[:-1,:] )/dy    # flux selon y
print("qx :", qx.shape, "(entre les colonnes)   qy :", qy.shape, "(entre les lignes)")

divx = ( qx[:,1:] - qx[:,:-1] )/dx                           # (ny, nx-2)
divy = ( qy[1:,:] - qy[:-1,:] )/dy                           # (ny-2, nx)
print("divx :", divx.shape, "  divy :", divy.shape, " <- tailles DIFFERENTES !")

dt = 0.01
S[:,1:-1] += dt*(-divx)      # on met a jour l'interieur SELON X
S[1:-1,:] += dt*(-divy)      # puis l'interieur SELON Y
print("S garde sa taille :", S.shape)

qx : (40, 59) (entre les colonnes)   qy : (39, 60) (entre les lignes)
divx : (40, 58)   divy : (38, 60)  <- tailles DIFFERENTES !
S garde sa taille : (40, 60)


`divx` et `divy` n'ont **pas la même taille** : on ne peut donc pas les additionner directement. La solution est de mettre à jour la solution en **deux temps**, chaque terme sur la zone qui lui correspond.

Dans la suite, nous emploierons l'une de ces trois techniques chaque fois qu'il faudra obtenir des tailles cohérentes pour les règles de mise à jour.

## À expérimenter

1. Essayez `S[1:-1,1:-1] += dt*(-divx - divy)`. Quelle erreur Python renvoie-t-il, et pourquoi est-ce une bonne nouvelle ?
2. Quelle est la taille de `0.5*(S[1:,1:] + S[:-1,:-1])` ? Devinez avant d'exécuter.